# Modul 7: Segmentasi citra dengan thresholding

Nama: Sahrul Ramadhani  
NIM: 244107020058  
Kelas: TI 3C

## Persiapan

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    drive.mount('/content/drive')
    IMAGES = Path('/content/drive/MyDrive/PCVK/Minggu 7/images')
else:
    lokasi = [Path.cwd() / 'images', Path.cwd() / 'Minggu 7' / 'images']
    IMAGES = Path(os.environ['PCVK_IMAGES']) if os.environ.get('PCVK_IMAGES') else next(
        (folder for folder in lokasi if folder.is_dir()), lokasi[0]
    )

print('Folder gambar:', IMAGES)

In [ ]:
import cv2 as cv
import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
from scipy import ndimage as ndi
from skimage import data, filters, color, segmentation, feature, morphology
from skimage.filters import gaussian
from skimage.segmentation import flood
from sklearn.cluster import KMeans
from IPython.display import display, Markdown

if DI_COLAB:
    from google.colab.patches import cv2_imshow

plt.rcParams['figure.dpi'] = 100

def baca_gambar(nama, grayscale=False):
    path = IMAGES / nama
    image = cv.imread(str(path), cv.IMREAD_GRAYSCALE if grayscale else cv.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f'Gambar tidak ditemukan atau tidak dapat dibaca: {path}')
    return image

def tampilkan(images, titles, rows=1, cols=None, figsize=(14, 5)):
    cols = len(images) if cols is None else cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize, squeeze=False)
    for ax, image, title in zip(axes.flat, images, titles):
        if image.ndim == 2:
            ax.imshow(image, cmap='gray')
        else:
            ax.imshow(cv.cvtColor(image, cv.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis('off')
    for ax in list(axes.flat)[len(images):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

KTP_INDIVIDU = 'ktp_1.jpg'

## C. Percobaan pada materi

### C.3 Global threshold dengan OpenCV

In [ ]:
img = baca_gambar('gradient.jpg')
thresh = 127
maxval = 255

ret, thresh1 = cv.threshold(img, thresh, maxval, cv.THRESH_BINARY)
ret, thresh2 = cv.threshold(img, thresh, maxval, cv.THRESH_BINARY_INV)
ret, thresh3 = cv.threshold(img, thresh, maxval, cv.THRESH_TRUNC)
ret, thresh4 = cv.threshold(img, thresh, maxval, cv.THRESH_TOZERO)
ret, thresh5 = cv.threshold(img, thresh, maxval, cv.THRESH_TOZERO_INV)

tampilkan(
    [img, thresh1, thresh2, thresh3, thresh4, thresh5],
    ['Original Image', 'BINARY', 'BINARY_INV', 'TRUNC', 'TOZERO', 'TOZERO_INV'],
    rows=2, cols=3, figsize=(15, 6)
)

### C.4 Adaptive threshold dengan OpenCV

In [ ]:
citra = cv.medianBlur(baca_gambar('sudoku-original.jpg'), 5)
gray = cv.cvtColor(citra, cv.COLOR_BGR2GRAY)
ret, th1 = cv.threshold(gray, 127, 255, cv.THRESH_BINARY)
th2 = cv.adaptiveThreshold(gray, 255, cv.ADAPTIVE_THRESH_MEAN_C, cv.THRESH_BINARY, 11, 2)
th3 = cv.adaptiveThreshold(gray, 255, cv.ADAPTIVE_THRESH_GAUSSIAN_C, cv.THRESH_BINARY, 11, 2)

tampilkan(
    [gray, th1, th2, th3],
    ['Citra asli', 'Global threshold (127)', 'Adaptive mean', 'Adaptive Gaussian'],
    rows=2, cols=2, figsize=(10, 10)
)

### C.5 Otsu dengan OpenCV

In [ ]:
img = baca_gambar('lena_gs_lc2.jpg', grayscale=True)
blur = cv.GaussianBlur(img, (5, 5), 0)
ret, th1 = cv.threshold(blur, 127, 255, cv.THRESH_BINARY)
ret2, th2 = cv.threshold(blur, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0, 0].hist(blur.ravel(), 256, [50, 200])
axes[0, 0].axvline(ret, color='red', label='Global = 127')
axes[0, 0].axvline(ret2, color='black', label=f'Otsu = {ret2:g}')
axes[0, 0].set_title('Histogram citra asli')
axes[0, 0].legend()
for ax, image, title in zip(
    list(axes.flat)[1:], [blur, th1, th2],
    ['Citra asli', 'Global threshold (127)', f'Otsu OpenCV ({ret2:g})']
):
    ax.imshow(image, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Threshold Otsu OpenCV:', ret2)

### C.5 Otsu sebelum dan sesudah Gaussian filter

In [ ]:
img = baca_gambar('noisy2.png', grayscale=True)
ret1, th1 = cv.threshold(img, 127, 255, cv.THRESH_BINARY)
ret2, th2 = cv.threshold(img, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
blur = cv.GaussianBlur(img, (5, 5), 0)
ret3, th3 = cv.threshold(blur, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for row, (source, result, source_title, result_title, threshold) in enumerate([
    (img, th1, 'Image noisy asli', 'Global threshold (127)', ret1),
    (img, th2, 'Image noisy asli', 'Otsu tanpa filter', ret2),
    (blur, th3, 'Image dengan Gaussian filter', 'Otsu setelah filter', ret3)
]):
    axes[row, 0].imshow(source, cmap='gray', vmin=0, vmax=255)
    axes[row, 0].set_title(source_title)
    axes[row, 0].axis('off')
    axes[row, 1].hist(source.ravel(), 256, [0, 256])
    axes[row, 1].set_title('Histogram')
    axes[row, 2].imshow(result, cmap='gray', vmin=0, vmax=255)
    axes[row, 2].set_title(f'{result_title}, T = {threshold:g}')
    axes[row, 2].axis('off')
plt.tight_layout()
plt.show()

### C.7 Segmentasi warna dengan K-Means

In [ ]:
img = cv.cvtColor(baca_gambar('lily.jpg'), cv.COLOR_BGR2RGB)
Z = np.float32(img.reshape((-1, 3)))
k = 3
kmeans = KMeans(n_clusters=k, random_state=42)
kmeans.fit(Z)
labels = kmeans.labels_
centers = np.uint8(kmeans.cluster_centers_)
segmented_img = centers[labels.flatten()].reshape(img.shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, image, title in zip(axes, [img, segmented_img], ['Citra asli', 'Segmentasi K-Means (k=3)']):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## D1. Tugas individu

### 3. Lima global threshold manual, T = 200

In [ ]:
gradient = baca_gambar('gradient.jpg', grayscale=True)
thresh = 200
maxval = 255

binary = np.where(gradient > thresh, maxval, 0).astype(np.uint8)
binary_inv = np.where(gradient > thresh, 0, maxval).astype(np.uint8)
trunc = np.where(gradient > thresh, thresh, gradient).astype(np.uint8)
tozero = np.where(gradient > thresh, gradient, 0).astype(np.uint8)
tozero_inv = np.where(gradient > thresh, 0, gradient).astype(np.uint8)

tampilkan(
    [gradient, binary, binary_inv, trunc, tozero, tozero_inv],
    ['Citra asli', 'BINARY', 'BINARY_INV', 'TRUNC', 'TOZERO', 'TOZERO_INV'],
    rows=2, cols=3, figsize=(15, 6)
)

### 4. Adaptive mean dan Gaussian manual

In [ ]:
def adaptive_manual(gray, block_size=11, C=2, sigma=2):
    if block_size < 3 or block_size % 2 == 0 or sigma <= 0:
        raise ValueError('Ukuran jendela harus ganjil, minimal 3; sigma harus positif.')
    height, width = gray.shape
    radius = block_size // 2
    padded = np.pad(gray.astype(np.float64), radius, mode='edge')

    posisi = np.arange(-radius, radius + 1, dtype=np.float64)
    xx, yy = np.meshgrid(posisi, posisi)
    kernel = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    kernel /= kernel.sum()

    mean = np.zeros((height, width), dtype=np.float64)
    gaussian_mean = np.zeros((height, width), dtype=np.float64)
    for y in range(block_size):
        for x in range(block_size):
            bagian = padded[y:y + height, x:x + width]
            mean += bagian
            gaussian_mean += bagian * kernel[y, x]
    mean /= block_size ** 2

    hasil_mean = np.where(gray > mean - C, 255, 0).astype(np.uint8)
    hasil_gaussian = np.where(gray > gaussian_mean - C, 255, 0).astype(np.uint8)
    return hasil_mean, hasil_gaussian

sudoku = baca_gambar('sudoku-original.jpg', grayscale=True)
global_manual = np.where(sudoku > 127, 255, 0).astype(np.uint8)
mean_manual, gaussian_manual = adaptive_manual(sudoku, block_size=11, C=2, sigma=2)
tampilkan(
    [sudoku, global_manual, mean_manual, gaussian_manual],
    ['Citra asli', 'Global manual (127)', 'Adaptive mean manual', 'Adaptive Gaussian manual'],
    rows=2, cols=2, figsize=(10, 10)
)

### 5A. Otsu manual pada Lena

In [ ]:
def otsu(gray):
    if gray.ndim != 2 or gray.dtype != np.uint8 or gray.size == 0:
        raise ValueError('Otsu memerlukan citra grayscale uint8 yang tidak kosong.')
    his, bins = np.histogram(gray, np.arange(0, 257))
    pixel_number = gray.size
    intensity_arr = np.arange(256)
    final_thresh = 0
    final_value = -1.0

    for t in range(256):
        pcb = np.sum(his[:t + 1])
        pcf = np.sum(his[t + 1:])
        if pcb == 0 or pcf == 0:
            continue
        Wb = pcb / pixel_number
        Wf = pcf / pixel_number
        mub = np.sum(intensity_arr[:t + 1] * his[:t + 1]) / pcb
        muf = np.sum(intensity_arr[t + 1:] * his[t + 1:]) / pcf
        value = Wb * Wf * (mub - muf) ** 2
        if value > final_value:
            final_thresh = t
            final_value = value

    final_img = np.where(gray > final_thresh, 255, 0).astype(np.uint8)
    return final_img, final_thresh

lena = baca_gambar('lena_gs_lc2.jpg', grayscale=True)
lena_blur = cv.GaussianBlur(lena, (5, 5), 0)
lena_otsu_manual, lena_thresh_manual = otsu(lena_blur)
lena_global = np.where(lena_blur > 127, 255, 0).astype(np.uint8)
lena_thresh_cv, lena_otsu_cv = cv.threshold(lena_blur, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

print('Threshold Otsu manual:', lena_thresh_manual)
print('Threshold Otsu OpenCV:', lena_thresh_cv)
print('Jumlah piksel yang berbeda:', np.count_nonzero(lena_otsu_manual != lena_otsu_cv))
tampilkan(
    [lena_blur, lena_global, lena_otsu_manual],
    ['Citra asli', 'Global threshold (127)', f'Otsu manual ({lena_thresh_manual})'],
    figsize=(15, 5)
)
tampilkan(
    [lena_otsu_manual, lena_otsu_cv], ['Otsu manual', 'Otsu OpenCV'], figsize=(10, 5)
)

#### Jawaban: hasil Lena pada threshold 127

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(lena_blur.ravel(), bins=256, range=(0, 256))
plt.axvline(127, color='red', label='Global = 127')
plt.axvline(lena_thresh_manual, color='black', label=f'Otsu = {lena_thresh_manual}')
plt.xlabel('Intensitas piksel')
plt.ylabel('Jumlah piksel')
plt.title('Histogram Lena')
plt.legend()
plt.tight_layout()
plt.show()

minimum, maksimum = int(lena_blur.min()), int(lena_blur.max())
print(f'Intensitas Lena berada pada {minimum} sampai {maksimum}.')
if maksimum <= 127:
    print('Semua piksel tidak lebih dari 127, sehingga binary threshold menghasilkan hitam seluruhnya.')
else:
    print('Ada piksel di atas 127, sehingga hasil tidak hitam seluruhnya pada gambar ini.')
print(f'Ambang {lena_thresh_manual} dari Otsu memisahkan bagian gelap dan terang Lena ini.')

### 5B. Otsu manual pada KTP

In [ ]:
ktp_asli = baca_gambar(KTP_INDIVIDU)
ktp_gray = cv.cvtColor(ktp_asli, cv.COLOR_BGR2GRAY)
ktp_blur = cv.GaussianBlur(ktp_gray, (5, 5), 0)
ktp_global = np.where(ktp_blur > 127, 255, 0).astype(np.uint8)
ktp_otsu_manual, ktp_thresh_manual = otsu(ktp_blur)
ktp_thresh_cv, ktp_otsu_cv = cv.threshold(ktp_blur, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

print('Threshold Otsu manual KTP:', ktp_thresh_manual)
print('Threshold Otsu OpenCV KTP:', ktp_thresh_cv)
print('Jumlah piksel yang berbeda:', np.count_nonzero(ktp_otsu_manual != ktp_otsu_cv))
tampilkan(
    [ktp_blur, ktp_global, ktp_otsu_manual],
    ['KTP grayscale', 'Global threshold (127)', f'Otsu manual ({ktp_thresh_manual})'],
    figsize=(15, 5)
)
tampilkan([ktp_otsu_manual, ktp_otsu_cv], ['Otsu manual', 'Otsu OpenCV'], figsize=(10, 5))
print(f'Global memakai ambang tetap 127. Otsu memilih ambang {ktp_thresh_manual} dari histogram KTP.')

### 6A. Region growing manual pada koin

Empat tetangga. Piksel masuk region jika selisih intensitasnya terhadap seed tidak lebih dari T. Koordinat seed ditulis sebagai (baris, kolom).

In [ ]:
def region_growing(gray, seed, tolerance):
    height, width = gray.shape
    row, col = seed
    if not (0 <= row < height and 0 <= col < width) or tolerance < 0:
        raise ValueError('Seed harus berada di dalam citra dan toleransi tidak boleh negatif.')
    seed_value = int(gray[row, col])
    mask = np.zeros(gray.shape, dtype=bool)
    visited = np.zeros(gray.shape, dtype=bool)
    queue = deque([seed])
    visited[seed] = True
    mask[seed] = True
    while queue:
        row, col = queue.popleft()
        for dy, dx in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            ny, nx = row + dy, col + dx
            if 0 <= ny < height and 0 <= nx < width and not visited[ny, nx]:
                visited[ny, nx] = True
                if abs(int(gray[ny, nx]) - seed_value) <= tolerance:
                    mask[ny, nx] = True
                    queue.append((ny, nx))
    return mask

coins = data.coins()
SEEDS = {'Dalam koin': (36, 43), 'Latar': (10, 10), 'Tepi koin': (35, 43)}
TOLERANSI = [5, 20, 60]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (label, (row, col)) in zip(axes, SEEDS.items()):
    ax.imshow(coins, cmap='gray', vmin=0, vmax=255)
    ax.scatter(col, row, s=60, marker='x', color='red')
    ax.set_title(f'{label}: ({row}, {col})')
    ax.axis('off')
plt.tight_layout()
plt.show()

region_manual = {}
jumlah_region_manual = []
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for row, (nama_seed, seed) in enumerate(SEEDS.items()):
    for col, T in enumerate(TOLERANSI):
        mask = region_growing(coins, seed, T)
        region_manual[(nama_seed, T)] = mask
        jumlah = int(mask.sum())
        jumlah_region_manual.append({'Seed': nama_seed, 'Koordinat': seed, 'T': T, 'Jumlah piksel': jumlah})
        overlay = color.label2rgb(mask.astype(np.uint8), image=coins, colors=['red'], bg_label=0, alpha=0.5)
        axes[row, col].imshow(overlay)
        axes[row, col].scatter(seed[1], seed[0], c='yellow', marker='x', s=25)
        axes[row, col].set_title(f'{nama_seed}, T={T}, piksel={jumlah}')
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()
display(pd.DataFrame(jumlah_region_manual))

### 6B. Region growing dengan skimage.segmentation.flood

In [ ]:
gray = data.coins()
seed = (121, 63)
region = flood(gray, seed_point=seed, tolerance=20, connectivity=1)
overlay = color.label2rgb(region.astype(np.uint8), image=gray, colors=['red'], bg_label=0, alpha=0.5)
tampilkan(
    [gray, (region * 255).astype(np.uint8), cv.cvtColor((overlay * 255).astype(np.uint8), cv.COLOR_RGB2BGR)],
    ['Citra asli', 'Mask flood, seed (121, 63), T=20', 'Region flood'], figsize=(15, 5)
)
print('Jumlah piksel flood, seed (121, 63), T=20:', int(region.sum()))

region_flood = {}
jumlah_region_flood = []
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for row, (nama_seed, seed) in enumerate(SEEDS.items()):
    for col, T in enumerate(TOLERANSI):
        mask = flood(gray, seed_point=seed, tolerance=T, connectivity=1)
        region_flood[(nama_seed, T)] = mask
        jumlah = int(mask.sum())
        jumlah_region_flood.append({'Seed': nama_seed, 'Koordinat': seed, 'T': T, 'Jumlah piksel': jumlah})
        overlay = color.label2rgb(mask.astype(np.uint8), image=gray, colors=['red'], bg_label=0, alpha=0.5)
        axes[row, col].imshow(overlay)
        axes[row, col].scatter(seed[1], seed[0], c='yellow', marker='x', s=25)
        axes[row, col].set_title(f'{nama_seed}, T={T}, piksel={jumlah}')
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()
display(pd.DataFrame(jumlah_region_flood))

### 7. Segmentasi warna biru KTP dengan K-Means

In [ ]:
ktp_rgb = cv.cvtColor(ktp_asli, cv.COLOR_BGR2RGB)
Z = np.float32(ktp_rgb.reshape((-1, 3)))
kmeans_ktp = KMeans(n_clusters=8, random_state=42)
labels_ktp = kmeans_ktp.fit_predict(Z)
centers_ktp = np.uint8(kmeans_ktp.cluster_centers_)
segmented_ktp = centers_ktp[labels_ktp].reshape(ktp_rgb.shape)

hsv_centers = cv.cvtColor(centers_ktp.reshape(1, -1, 3), cv.COLOR_RGB2HSV).reshape(-1, 3)
H, S, V = hsv_centers[:, 0], hsv_centers[:, 1], hsv_centers[:, 2]
kelompok_biru = np.where((H >= 85) & (H <= 130) & (S >= 25) & (V >= 40))[0]
mask_biru = np.isin(labels_ktp, kelompok_biru).reshape(ktp_gray.shape)
hanya_biru = np.zeros_like(ktp_rgb)
hanya_biru[mask_biru] = ktp_rgb[mask_biru]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, image, title in zip(
    axes, [ktp_rgb, segmented_ktp, hanya_biru],
    ['Citra asli', 'Segmentasi K-Means (K=8)', 'Hanya warna biru']
):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Cluster biru:', kelompok_biru.tolist())
print('Jumlah piksel biru:', int(mask_biru.sum()))

## D2. Tugas kelompok: perbandingan OCR KTP

### Instalasi Tesseract dan bahasa Indonesia

In [ ]:
import importlib.util
import subprocess
import sys
import shutil

if DI_COLAB:
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'tesseract-ocr', 'tesseract-ocr-ind'], check=True)
if importlib.util.find_spec('pytesseract') is None:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'pytesseract'], check=True)

import pytesseract
from pytesseract import Output

tesseract_path = os.environ.get('PCVK_TESSERACT') or shutil.which('tesseract')
if tesseract_path is None and os.name == 'nt':
    pilihan = [Path('C:/Program Files/Tesseract-OCR/tesseract.exe'),
               Path('C:/Program Files/PDF24/tesseract/tesseract.exe')]
    tesseract_path = next((str(path) for path in pilihan if path.is_file()), None)
if tesseract_path is None:
    raise FileNotFoundError('Tesseract OCR belum terpasang.')
pytesseract.pytesseract.tesseract_cmd = tesseract_path

BAHASA_OCR = 'ind'
PSM_OCR = 6
BATAS_CONFIDENCE = 60
if os.environ.get('PCVK_TESSDATA'):
    os.environ['TESSDATA_PREFIX'] = os.environ['PCVK_TESSDATA']
CONFIG_OCR = f'--psm {PSM_OCR}'
if BAHASA_OCR not in pytesseract.get_languages():
    raise RuntimeError('Model bahasa Indonesia (ind) belum tersedia pada Tesseract.')
print('Bahasa:', BAHASA_OCR, '| PSM:', PSM_OCR, '| Kotak: confidence >', BATAS_CONFIDENCE)

### 1. Tiga citra berukuran sama untuk setiap anggota

In [ ]:
NAMA_ANGGOTA = {'ktp_1.jpg': 'Sahrul Ramadhani'}
file_ktp = sorted(
    (path for path in IMAGES.iterdir()
     if path.is_file() and path.stem.lower().startswith('ktp_')
     and path.suffix.lower() in {'.jpg', '.jpeg', '.png'}),
    key=lambda path: path.name.lower()
)
if not file_ktp:
    raise FileNotFoundError(f'File ktp_1.jpg dan KTP anggota belum ada di {IMAGES}')

citra_ocr = {}
for path in file_ktp:
    nama = NAMA_ANGGOTA.get(path.name, path.stem)
    original = baca_gambar(path.name)
    grayscale = cv.cvtColor(original, cv.COLOR_BGR2GRAY)
    _, otsu_binary = cv.threshold(grayscale, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    adaptive_binary = cv.adaptiveThreshold(
        grayscale, 255, cv.ADAPTIVE_THRESH_GAUSSIAN_C, cv.THRESH_BINARY, 11, 2
    )
    citra_ocr[path.name] = {
        'nama': nama,
        'images': {'Berwarna asli': original, 'Segmentasi Otsu': otsu_binary,
                   'Segmentasi Adaptive': adaptive_binary}
    }
    display(Markdown(f'#### {nama} ({path.name})'))
    tampilkan([original, otsu_binary, adaptive_binary], ['Berwarna', 'Otsu', 'Adaptive'], figsize=(18, 5))
    print('Ukuran ketiga citra:', original.shape[:2])

### 2 dan 3. Teks OCR dan kotak hijau, confidence > 60

In [ ]:
hasil_ocr = {}
for filename, anggota in citra_ocr.items():
    hasil_ocr[filename] = {'nama': anggota['nama'], 'metode': {}}
    kotak_images = []
    kotak_titles = []
    display(Markdown(f"#### {anggota['nama']} ({filename})"))
    for metode, image in anggota['images'].items():
        ocr_input = cv.cvtColor(image, cv.COLOR_BGR2RGB) if image.ndim == 3 else image
        teks = pytesseract.image_to_string(ocr_input, lang=BAHASA_OCR, config=CONFIG_OCR, timeout=120)
        detail = pytesseract.image_to_data(
            ocr_input, lang=BAHASA_OCR, config=CONFIG_OCR, output_type=Output.DICT, timeout=120
        )
        marked = image.copy() if image.ndim == 3 else cv.cvtColor(image, cv.COLOR_GRAY2BGR)
        confidence_words = []
        for i, kata in enumerate(detail['text']):
            confidence = float(detail['conf'][i])
            if str(kata).strip() and confidence >= 0:
                confidence_words.append(confidence)
                if confidence > BATAS_CONFIDENCE:
                    x, y, w, h = (int(detail[key][i]) for key in ['left', 'top', 'width', 'height'])
                    cv.rectangle(marked, (x, y), (x + w, y + h), (0, 255, 0), 2)
        rata_confidence = float(np.mean(confidence_words)) if confidence_words else 0.0
        hasil_ocr[filename]['metode'][metode] = {
            'teks': teks, 'data': detail, 'rata_confidence': rata_confidence, 'kotak': marked
        }
        kotak_images.append(marked)
        kotak_titles.append(metode + ', confidence > 60')
        print()
        print(metode)
        print(teks)
    tampilkan(kotak_images, kotak_titles, figsize=(18, 5))

### 4. Tabel 10 kata atau angka pembanding

Acuan berupa 10 kata pada label KTP. Penilaian memakai teks OCR lengkap. Kata yang mirip tetap dihitung salah jika hurufnya tidak sama dengan acuan.

In [ ]:
from difflib import SequenceMatcher

ACUAN_KTP = ['NIK', 'Nama', 'Alamat', 'Agama', 'Status', 'Perkawinan',
             'Pekerjaan', 'Kewarganegaraan', 'Berlaku', 'Hingga']

def pasangan_ocr(acuan, teks):
    kata_ocr = teks.split()
    bacaan = [''] * len(acuan)
    terpakai = set()

    for row, kata_asli in enumerate(acuan):
        for col, kata in enumerate(kata_ocr):
            if col not in terpakai and kata_asli.strip().lower() == kata.strip().lower():
                bacaan[row] = kata
                terpakai.add(col)
                break

    kandidat = []
    for row, kata_asli in enumerate(acuan):
        if bacaan[row]:
            continue
        for col, kata in enumerate(kata_ocr):
            if col not in terpakai:
                kemiripan = SequenceMatcher(None, kata_asli.lower(), kata.lower()).ratio()
                if kemiripan >= 0.65:
                    kandidat.append((kemiripan, row, col))
    for _, row, col in sorted(kandidat, key=lambda item: (-item[0], item[1], item[2])):
        if not bacaan[row] and col not in terpakai:
            bacaan[row] = kata_ocr[col]
            terpakai.add(col)
    return bacaan

def isi_tabel_ocr(hasil):
    tabel = pd.DataFrame({'No.': range(1, 11), 'Teks sebenarnya': ACUAN_KTP})
    for metode, kolom in [('Berwarna asli', 'OCR citra berwarna'),
                          ('Segmentasi Otsu', 'OCR Otsu'),
                          ('Segmentasi Adaptive', 'OCR Adaptive')]:
        tabel[kolom] = pasangan_ocr(ACUAN_KTP, hasil['metode'][metode]['teks'])
    return tabel

tabel_perbandingan = {}
for filename, hasil in hasil_ocr.items():
    display(Markdown(f"#### {hasil['nama']} ({filename})"))
    tabel_perbandingan[filename] = isi_tabel_ocr(hasil)
    display(tabel_perbandingan[filename])

### 5. Persentase kecocokan dan rata-rata confidence

In [ ]:
def cocok(acuan, bacaan):
    return bool(bacaan.strip()) and acuan.strip().lower() == bacaan.strip().lower()

ringkasan_ocr = {}
for filename, tabel in tabel_perbandingan.items():
    rows = []
    for metode, kolom in [('Berwarna asli', 'OCR citra berwarna'),
                          ('Segmentasi Otsu', 'OCR Otsu'),
                          ('Segmentasi Adaptive', 'OCR Adaptive')]:
        jumlah_cocok = sum(cocok(acuan, bacaan) for acuan, bacaan in zip(tabel['Teks sebenarnya'], tabel[kolom]))
        rows.append({
            'Citra masukan OCR': metode,
            'Item terbaca tepat': jumlah_cocok,
            'Kecocokan (%)': jumlah_cocok / 10 * 100,
            'Rata-rata confidence': hasil_ocr[filename]['metode'][metode]['rata_confidence']
        })
    ringkasan_ocr[filename] = pd.DataFrame(rows)
    display(Markdown(f"#### {hasil_ocr[filename]['nama']} ({filename})"))
    display(ringkasan_ocr[filename].round(2))

### Jawaban pertanyaan analisis

In [ ]:
def analisis_ocr(tabel, ringkasan):
    terbaik = ringkasan[ringkasan['Kecocokan (%)'] == ringkasan['Kecocokan (%)'].max()]
    daftar_terbaik = ', '.join(terbaik['Citra masukan OCR'])
    bukti = terbaik.iloc[0]
    print(f"1. Hasil paling tepat: {daftar_terbaik}, dengan {int(bukti['Item terbaca tepat'])}/10 item cocok ({bukti['Kecocokan (%)']:g}%).")

    kesalahan = []
    for _, row in tabel.iterrows():
        for kolom in ['OCR citra berwarna', 'OCR Otsu', 'OCR Adaptive']:
            if not cocok(row['Teks sebenarnya'], row[kolom]):
                if row[kolom]:
                    kesalahan.append(f"{kolom}: acuan {row['Teks sebenarnya']}, kata OCR terdekat {row[kolom]} (tidak cocok).")
                else:
                    kesalahan.append(f"{kolom}: kata {row['Teks sebenarnya']} tidak ditemukan pada teks OCR.")
                break
    if len(kesalahan) >= 2:
        print('2. ' + ' '.join(kesalahan[:2]))
    elif kesalahan:
        print('2. Hanya ada satu kesalahan pada 10 acuan: ' + kesalahan[0])
    else:
        print('2. Tidak ditemukan kesalahan pada 10 acuan di ketiga metode.')

    skor = ringkasan.set_index('Citra masukan OCR')['Kecocokan (%)']
    asli = skor['Berwarna asli']
    otsu_skor = skor['Segmentasi Otsu']
    adaptive_skor = skor['Segmentasi Adaptive']
    if otsu_skor > asli and adaptive_skor > asli:
        simpulan = 'Kedua segmentasi meningkatkan kecocokan pada percobaan ini.'
    elif otsu_skor <= asli and adaptive_skor <= asli:
        simpulan = 'Kedua segmentasi tidak meningkatkan kecocokan pada percobaan ini.'
    else:
        simpulan = 'Hanya salah satu segmentasi meningkatkan kecocokan pada percobaan ini.'
    print(f'3. Segmentasi tidak selalu meningkatkan OCR. {simpulan} Berwarna {asli:g}%, Otsu {otsu_skor:g}%, Adaptive {adaptive_skor:g}%.')
    print('4. Warna dan pola latar dapat mengurangi perbedaan antara huruf dan latar. Threshold dapat memperjelas huruf, tetapi pola latar juga dapat menjadi bintik atau garis yang mengganggu bacaan.')

    tertinggi_conf = ringkasan[ringkasan['Rata-rata confidence'] == ringkasan['Rata-rata confidence'].max()]
    nama_conf = set(tertinggi_conf['Citra masukan OCR'])
    nama_akurat = set(terbaik['Citra masukan OCR'])
    if nama_conf <= nama_akurat:
        hubungan = 'Ya, pada percobaan ini semua metode dengan confidence tertinggi juga memiliki kecocokan tertinggi.'
    elif nama_conf & nama_akurat:
        hubungan = 'Hanya sebagian metode yang memiliki confidence tertinggi juga memiliki kecocokan tertinggi.'
    else:
        hubungan = 'Tidak, metode dengan confidence tertinggi tidak memiliki kecocokan tertinggi.'
    bukti_conf = '; '.join(
        f"{row['Citra masukan OCR']}: confidence {row['Rata-rata confidence']:.2f}, cocok {row['Kecocokan (%)']:g}%"
        for _, row in tertinggi_conf.iterrows()
    )
    print('5. ' + hubungan + ' ' + bukti_conf + '. Confidence adalah perkiraan mesin; kecocokan diperiksa dari tulisan asli.')

for filename, tabel in tabel_perbandingan.items():
    display(Markdown(f"#### {hasil_ocr[filename]['nama']} ({filename})"))
    analisis_ocr(tabel, ringkasan_ocr[filename])